In [0]:
%python
from datetime import datetime

# === CONFIGURATION ===
catalog = "bronze_dev"
schema = "ecommerce"
volume = "volumes/raw_data"  # note: includes the extra 'volumes/' segment

# === FUNCTION TO BUILD RAW PATH DYNAMICALLY ===
def build_source_path(table_name: str) -> str:
    """
    Derive the raw volume path for a given bronze table.
    Assumes: ecommerce_customers_base_metrics → /Volumes/bronze_dev/ecommerce/volumes/raw_data/customers_base_metrics/
    """
    # Extract base segment after "ecommerce_"
    if table_name.startswith("ecommerce_"):
        base_name = table_name.replace("ecommerce_", "")
    else:
        base_name = table_name

    return f"/Volumes/{catalog}/{schema}/{volume}/{base_name}/"

# === BRONZE TABLES LIST ===
bronze_tables = [
    "ecommerce_customers_base_metrics",
    "ecommerce_customers_demographics",
    "ecommerce_event_triggers",
    "ecommerce_invoice_customer_bridge",
    "ecommerce_invoice_items",
    "ecommerce_journey_events",
    "ecommerce_marketing_campaigns",
    "ecommerce_marketing_spend",
    "ecommerce_products_dim",
]

# === EMPTY LIST TO COLLECT LOGS ===
run_log = []

# === LOOP THROUGH TABLES ===
for table_name in bronze_tables:
    source_path = build_source_path(table_name)
    print(f"🔍 Validating {table_name} from {source_path}")

    # Count records in raw source (Parquet)
    try:
        source_count = spark.read.format("parquet").load(source_path).count()
    except Exception as e:
        source_count = None
        print(f"⚠️ Could not read source for {table_name}: {e}")

    # Count records in bronze table
    try:
        target_count = spark.table(f"{catalog}.{schema}.{table_name}").count()
    except Exception as e:
        target_count = None
        print(f"⚠️ Could not read bronze table {table_name}: {e}")

    # Validate if counts match
    validation = (
        source_count == target_count
        if (source_count is not None and target_count is not None)
        else False
    )

    # Add to log
    run_log.append(
        (table_name, datetime.now(), source_count, target_count, validation)
    )

# === CONVERT TO DATAFRAME ===
run_log_df = spark.createDataFrame(
    run_log,
    ["table_name", "timestamp", "source_count", "target_count", "validation_passed"]
)

# === WRITE OR APPEND TO RUN LOG TABLE ===
run_log_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.run_log")

print("✅ Run log updated successfully.")


In [0]:
%sql
select * from bronze_dev.ecommerce.run_log;